## naver_news 테이블의 한 행을 표현하기 위한 클래스 생성

In [20]:
# 네이버 뉴스 응답 데이터를 담을 NaverNews 엔티티 클래스 정의
from datetime import datetime

class NaverNews:

    def __init__(self, id: int, title: str, originallink: str, link: str, description: str, pub_date: str, created_at: datetime = None):
        self.__id = id
        self.__title = title
        self.__originallink = originallink
        self.__link = link
        self.__description = description
        self.__pub_date = pub_date
        self.__created_at = created_at

    @property
    def id(self):
        return self.__id

    @property
    def title(self):
        return self.__title

    # __title 속성에 대한 setter 메소드 예시
    # 아래 주석을 해제하면 naver_news.title = "새 제목"처럼 값을 변경할 수 있다.
    # @title.setter
    # def title(self, value):
    #     self.__title = value

    @property
    def originallink(self):
        return self.__originallink

    @property
    def link(self):
        return self.__link

    @property
    def description(self):
        return self.__description

    @property
    def pub_date(self):
        return self.__pub_date

    @property
    def created_at(self):
        return self.__created_at

    # naver news api 응답 데이터를 NaverNews 객체로 변환하기 위한 클래스 메서드
    @classmethod
    def from_api_item(cls, item:dict):
        return cls(
            id=None,
            title=item.get('title'),
            originallink=item.get('originallink'),
            link=item.get('link'),
            description=item.get('description'),
            pub_date=item.get('pubDate'),
        )

    def __repr__(self):
        # 객체를 print() 했을 때 어떤 값이 들어 있는지 확인하기 위한 문자열 표현 메소드
        return f'NaverNews({self.__id}, {self.__title}, {self.__originallink}, {self.__link}, {self.__description}, {self.__pub_date}, {self.__created_at})'

## .env를 읽어서 환경변수 등록

In [27]:
from dotenv import load_dotenv
import os

load_dotenv()

NAVER_CLIENT_ID = os.getenv("NAVER_CLIENT_ID")
NAVER_CLIENT_SECRET = os.getenv("NAVER_CLIENT_SECRET")

headers = {
    "X-Naver-Client-Id": NAVER_CLIENT_ID,
    "X-Naver-Client-Secret": NAVER_CLIENT_SECRET
}

config = {
    "host": os.getenv("DB_HOST", "localhost"), # (key, default)
    "port": os.getenv("DB_PORT", "3306"),  # 기본포트 3306인 경우 생략가능
    "user": os.getenv("DB_USER", "skn_ai"),
    "password": os.getenv("DB_PASSWORD", "1234"),
    "database": os.getenv("DB_DATABASE", "menudb")
}

if not NAVER_CLIENT_ID or not NAVER_CLIENT_SECRET:
    raise ValueError('NAVER_CLIENT_ID 또는 NAVER_CLIENT_SECRET이 설정되지 않았습니다.')

## 네이버 뉴스 - "인공지능" 관련 최근 뉴스 10개 조회해서 DB에 저장

In [43]:
import requests
from pprint import pprint
# 출력 시 공백문자를 통해 가독성 있게 출력함

url = 'https://openapi.naver.com/v1/search/news.json'

params = {
    'query' : '인공지능',
    'display' : '10',
    'start' : 1,
    'sort' : 'date'
}

naver_news_list: list[NaverNews] = []

try:
    # GET Method == 조회 요청
    response = requests.get(
        url,  # 인자가 너무 많아 url을 제외한 나머지는 키워드 인자로 값을 넣어줌
        headers=headers,
        params=params,  # dict -> 쿼리 스트링 변환 (url encoding)
        timeout=10
    )

    # HTTP 상태 코드가 400, 500번대인 경우 예외 발생
    response.raise_for_status() # raise: '에러를 던지겠다'

    response_code = response.status_code    # 상태 코드
    data = response.json() # 응답 데이터 (json) -> dict 변환

    # pprint(data)  # list[dict]
    # 응답 데이터 중 뉴스 기사 리스트인 'items'만 사용
    items = data.get('items',[])
    # pprint(items)
    naver_news_list = [NaverNews.from_api_item(item) for item in items]

    print(naver_news_list)


except requests.exceptions.Timeout:
    print("요청 시간 10초 초과")
except ValueError:
    print("응답 데이터가 JSON 형식이 아닙니다")

[NaverNews(None, 충남 'AI 대전환' 올인…5.8兆 쏟아붓는다, https://www.hankyung.com/article/2026061575971, https://n.news.naver.com/mnews/article/015/0005298916?sid=102, 충청남도가 제조업 중심의 산업 구조를 <b>인공지능</b>(AI) 기반 산업 체계로 바꾸는 ‘충남 AI 대전환’에 속도를 내고 있다. 도는 2035년까지 5조8900억원을 투입해 제조업 AI 전환(AX)과 데이터센터 구축, AI 인재 양성을... , Mon, 15 Jun 2026 17:23:00 +0900, None), NaverNews(None, 신한은행, 체험형 인턴십 운영... 우수 수료자 채용 우대, http://www.thefirstmedia.net/news/articleView.html?idxno=201362, http://www.thefirstmedia.net/news/articleView.html?idxno=201362, 최근 금융권 전반에서 디지털 전환과 <b>인공지능</b>(AI) 활용이 확대됨에 따라 디지털 역량과 AI 활용 능력, 협업 및 커뮤니케이션 역량을 강화하는 실습형 교육을 확대하고 있다. 현업 멘토링 프로그램도 함께 운영해 신입... , Mon, 15 Jun 2026 17:22:00 +0900, None), NaverNews(None, 비즈데이터, 수처리 공정 XAI 시스템으로 K-water 성과공유제 선정, https://www.mt.co.kr/industry/2026/06/15/2026061514414969916, https://n.news.naver.com/mnews/article/008/0005372190?sid=101, 비즈데이터에 따르면, 이번 선정 과제는 '설명가능한 <b>인공지능</b>(XAI) 기반 수처리 약품 공정 의사결정 시스템' 개발이다. 해당 시스템은 경기도 안산시 시흥·반월 정수장에 구축돼 성능 검증을 완료했다. 성과공유제는... ,

In [33]:
import mysql.connector

try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            for naver_news in naver_news_list:
                cursor.execute('''
                    insert into naver_news (
                        title,
                        originallink,
                        link,
                        description,
                        pub_date)
                    values (%s, %s, %s, %s, %s)
                ''', (
                    naver_news.title,
                    naver_news.originallink,
                    naver_news.link,
                    naver_news.description,
                    naver_news.pub_date
                ))
            conn.commit()   # 전체 insert 내용 커밋

except mysql.connector.Error as err:
    print("DB 에러: ", err)
    conn.rollback() # DB 관련하여 오류 발생 시 전체 rollback

In [37]:
with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            cursor.execute("select * from naver_news")
            a = cursor.fetchall()
            pprint(a)

[(1,
  '충격에 빠진 국내 고용 시장... 무려 316개월 만에 감소세로 전환된 주요...',
  'https://www.wikitree.co.kr/articles/1141496',
  'https://www.wikitree.co.kr/articles/1141496',
  '<b>인공지능</b>(AI)이 단순 반복적인 코딩을 대체하면서 진입 장벽이 높아졌고 시스템 설계를 맡는 숙련직 수요만 유지되는 '
  '양상이다. 30대 상용직은 전문·과학·기술 서비스업에서 7만 6000명 줄어 전 업종 통틀어 가장... ',
  'Mon, 15 Jun 2026 16:40:00 +0900',
  datetime.datetime(2026, 6, 15, 16, 50, 25)),
 (2,
  '건강한 디지털 양육 문화 조성...동대문구, 학부모 대상 특강 운영',
  'https://www.hg-times.com/news/articleView.html?idxno=302763',
  'https://www.hg-times.com/news/articleView.html?idxno=302763',
  '구는 &quot;강의는 <b>인공지능</b>(AI) 기술 보편화에 따른 AI 리터러시 향상, 스마트폰 사용 문제를 둘러싼 부모와 자녀 '
  '간 갈등 완화, 가정 내 건강한 소통 관계 형성에 기여한다&quot;고 설명했다. 특강은 오는 25일 목요일 오후 7시부터... ',
  'Mon, 15 Jun 2026 16:40:00 +0900',
  datetime.datetime(2026, 6, 15, 16, 50, 25)),
 (3,
  '현대로템, 佛서 첨단 방호체계 기술 글로벌 첫선',
  'http://www.hansbiz.co.kr/news/articleView.html?idxno=844399',
  'http://www.hansbiz.co.kr/news/articleView.html?idxno=844399',
  "현대로템 전시관 전경./현대로템 | 서울=한스경제 임준혁 

In [38]:
# DB에 저장된 네이버 뉴스 데이터를 조회해 객체 리스트로 변환
try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            cursor.execute('''
                           select *
                           from naver_news
                           order by id desc
                           ''')  # sql, params
            naver_news_list = [NaverNews(*row) for row in cursor.fetchall()]
            print(naver_news_list)
except mysql.connector.Error as e:
    print('DB 오류:', e)

[NaverNews(10, 트럼프 정부 반도체 이어 'AI 모델 접근권'까지 통제, 중국 딥시크에 빈..., https://www.businesspost.co.kr/BP?command=article_view&num=439990, https://www.businesspost.co.kr/BP?command=article_view&num=439990, &lt;연합뉴스&gt; 미국 도널드 트럼프 정부가 앤트로픽의 최신 <b>인공지능</b>(AI) 모델에 외국인의 접근을 제한하면서 AI와 관련한 통제 범위가 반도체에서 AI 모델 자체로 확대할 조짐을 보이고 있다. 동맹국까지 대상에 포함한... , Mon, 15 Jun 2026 16:40:00 +0900, 2026-06-15 16:50:25), NaverNews(9, [오늘의증시] 코스피, 중동 훈풍에 5%대 급등…8,500선 회복, https://news.ifm.kr/news/articleView.html?idxno=473049, https://news.ifm.kr/news/articleView.html?idxno=473049, 친환경 선박과 <b>인공지능</b> 데이터센터 수요 확대 기대가 맞물리며 한화엔진은 10.0%, HD현대마린엔진은 7.6% 상승했다. 대형주 중에서는 HD현대중공업이 9.9%, LS ELECTRIC이 15.7% 뛰었다. 청년 탈모 치료 건강보험 적용 추진... , Mon, 15 Jun 2026 16:40:00 +0900, 2026-06-15 16:50:25), NaverNews(8, 경남도, 흩어진 관광 데이터 모아 'AI 관광 통합플랫폼' 만든다, https://www.etnews.com/20260615000421, https://n.news.naver.com/mnews/article/030/0003437833?sid=102, 특히 경남도는 이번 공모에서 기존 데이터 허브에 분산된 관광 데이터를 연계하고 <b>인공지능</b>(AI) 기반 서비스를 접목하는 방안에서 우수성을 인정받았다. 경남도

## 중복 뉴스 방지
- 중복 뉴스(중복 행)를 방지하기 위해서는 중복 기준이 되는 컬럼을 설정하고, 컬럼 값이 중복이면 update가 수행되게 해야 함.
    - 중복 판단을 하려면 PK 또는 UNIQUE 제약조건을 이용 (UNIQUE: 중복되는 걸 알아서 찾음 --> 빠른 중복 판단 가능)

- `replace` 대신 `on duplicate key update` 구문 사용
    - replace: delete -> insert
    - on duplicate key update: update 1회

```sql
alter table naver_news
add constraint uq_naver_news_link unique (link)     # link는 중복될 수 없기에 uniqeu 요소로 설정
```

In [45]:
insert_count = 0
skip_count = 0

try:
    with mysql.connector.connect(**config) as conn:
        with conn.cursor() as cursor:
            for naver_news in naver_news_list:
                cursor.execute('''
                    insert into naver_news
                        (title, originallink, link, description, pub_date)
                    values
                        (%s, %s, %s, %s, %s)
                    on duplicate key update
                        link = link
                ''', (
                    naver_news.title,
                    naver_news.originallink,
                    naver_news.link,
                    naver_news.description,
                    naver_news.pub_date
                ))

                if cursor.rowcount == 1:
                    insert_count += 1       # '업데이트'에 대한 건 insert_count가 증가하지 않음. '0'.
                else:
                    skip_count += 1

            conn.commit()

    print(f"신규 저장: {insert_count}건")
    print(f"중복 제외: {skip_count}건")

except mysql.connector.Error as e:
    print('DB 오류:', e)

신규 저장: 0건
중복 제외: 10건
